At the end of the mini-project, you will be able to:

* Understand how Agentic AI can automate core banking operations
* Design tool-based systems for LLM interaction with structured data
* Build agents that interpret user queries and invoke appropriate tools
* Develop orchestration workflows between agents, tools, and databases
* Implement secure transaction handling with basic validations
* Generate structured outputs such as mini statements
* Handle filtered queries like debit/credit transaction views
* Test and validate the system using real-world scenarios


## Problem Statement

Banks and financial institutions handle a large volume of customer requests related to account information, transactions, and fund transfers. These interactions are often initiated through chat, mobile apps, or support channels and involve repetitive yet critical operations.

Currently, many banking support processes are **manual, fragmented, and query-driven**, leading to several challenges:

* **Delayed response times** for customer requests
* **Inconsistent handling** of similar queries across channels
* **Over-reliance on human intervention** for routine operations
* **Limited ability to process natural language queries efficiently**
* **Lack of seamless integration** between user interactions and backend systems

A significant portion of customer requests typically falls into the following categories:

* **Information retrieval**, such as checking account balance or viewing transactions
* **Transaction operations**, such as transferring funds between accounts
* **Filtered queries**, such as viewing only debit or credit transactions
* **Statement generation**, such as requesting a mini statement of recent activities

However, most existing systems lack the capability to:

* Automatically **understand and interpret user queries from natural language inputs**
* Dynamically **select and execute appropriate backend operations (tools)**
* Provide **consistent, accurate, and real-time responses**
* Seamlessly **integrate conversational AI with structured banking data**

*The goal is to design and develop an **AI-Based Agentic Banking System** that can:*

* Interpret user queries and map them to appropriate banking operations
* Enable intelligent agents to interact with tools for balance checks, transactions, and transfers
* Support filtered views of transaction data (debit/credit)
* Generate structured outputs such as mini statements
* Provide a unified, automated, and efficient banking assistance experience


## Database Tables

Design a structured database to support all banking operations and tool interactions.

* ***customers***

  Stores basic customer details such as customer ID, name, and contact information (email).

* ***accounts***

  Maintains account-specific data including account ID, linked customer ID, and current balance.

* ***transactions***

  Records all debit and credit transactions with details like transaction ID, account ID, amount, type, and timestamp.

These tables enable efficient data retrieval, transaction processing, and support all agent-driven banking functionalities.


## Sample Input Queries to Test the System

1. Check Balance

    * *What is my current balance?*
    * *Show my account balance*
    * *How much money do I have in my account?*

---

2. See Transactions

    * *Show my recent transactions*
    * *Display last 10 transactions*
    * *Give me my transaction history*

---

3. Filter Debit/Credit

    * *Show only debit transactions*
    * *Show only credit transactions*
    * *List all my debit entries*
    * *Display my credit transactions*

---

4. Transfer Amount

    * *Transfer ₹2000 to ACC1002*
    * *Pay ₹1000 to account number ACC1002*

---

5. Mini Statement

    * *Generate my mini statement*
    * *Show last 5 transactions as a statement*
    * *Give me my mini statement for recent transactions*

---

6. Edge Case / Validation Queries

    * *Transfer ₹1,00,000 to ACC1002 (insufficient balance scenario)*
    * *Send money to an unknown user*
    * *Show transactions for last 2 years*
    * *Transfer money without specifying amount*

---

### Install Required Libraries

In [1]:
%%capture
!pip -q install openai==2.3.0
!pip -q install langchain-core==0.3.79
!pip -q install langchain-community==0.3.31
!pip -q install sentence-transformers==5.1.1
!pip -q install langchain-huggingface==0.3.1
!pip -q install langchain-experimental==0.3.4
!pip -q install langchainhub==0.1.21
!pip -q install langchain-openai==0.3.35
!pip -q install langgraph==0.6.8

### Import Neccesary Packages

In [2]:
import os
import sqlite3
from datetime import datetime
from google.colab import userdata

from langchain_core.tools import tool
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, create_react_agent, tools_condition
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages

from typing import TypedDict, List, Optional, Annotated
from langchain_core.messages import BaseMessage
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.checkpoint.memory import MemorySaver

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


### Read the Groq API Key

Using the Groq API Key, you will have access to the OpenAI and Llama model, free of cost, under the [Free-tier](https://console.groq.com/docs/rate-limits#rate-limits).

This cell retrieves the Groq API key securely stored in Google Colab's Secrets (see 'key' icon on the leftmost panel of this notebook) and sets it as an environment variable `GROQ_API_KEY`. This is a recommended practice to avoid exposing your API key directly in the code.


* Go to https://console.groq.com/, and setup a Free account.

* Create a new API by visiting: https://console.groq.com/keys

* Save the key in Google Colab's Secrets
    ```
    Secret Name: GROQ_API_KEY
    Secret Value: Paste your Groq api key
    ```

* Read the key and save as an environment variable `GROQ_API_KEY`

In [3]:
# Save the key in Colab's Secrets then load from there

import os
from google.colab import userdata

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

### Initialize LLM

***Note:** Groq models have [rate limits](https://console.groq.com/docs/rate-limits)—you can use alternative models from other providers if required.*

In [4]:
llm = ChatOpenAI(
    model="openai/gpt-oss-120b",   # Groq-supported model
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

### **Create a SQLite Database (in-memory)**

**SQLite** is a C library that provides a lightweight disk-based database that doesn't require a separate server process and allows accessing the database using a nonstandard variant of the SQL query language. Some applications can use SQLite for internal data storage.

SQLite3 specifically refers to the third version of SQLite.

*You may use the provided sample database created below and modify the tables or data if required.*

In [5]:
import sqlite3
# --- SQLite DB ---
DB_FILE = "banking_system.db"


def init_db():

    # Create / connect to SQLite database
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    # Enable foreign key support
    cursor.execute("PRAGMA foreign_keys = ON;")

    # -----------------------------
    # Create Tables
    # -----------------------------

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS customers (
        customer_id TEXT PRIMARY KEY,
        name TEXT NOT NULL,
        email TEXT,
        phone TEXT
    );
    """)

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS accounts (
        account_id TEXT PRIMARY KEY,
        customer_id TEXT NOT NULL,
        account_number TEXT UNIQUE NOT NULL,
        balance REAL NOT NULL,
        FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
    );
    """)

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS transactions (
        transaction_id TEXT PRIMARY KEY,
        account_id TEXT NOT NULL,
        transaction_type TEXT NOT NULL CHECK(transaction_type IN ('debit', 'credit')),
        amount REAL NOT NULL,
        description TEXT,
        transaction_timestamp TEXT NOT NULL,
        FOREIGN KEY (account_id) REFERENCES accounts(account_id)
    );
    """)

    # -----------------------------
    # Insert Sample Data
    # -----------------------------

    customers_data = [
        ("C1", "Aarav Sharma", "aarav@example.com", "9876543210"),
        ("C2", "Rahul Verma", "rahul@example.com", "9876501234"),
        ("C3", "Priya Mehta", "priya@example.com", "9876512345"),
        ("C4", "Neha Kapoor", "neha@example.com", "9876523456"),
        ("C5", "Rohan Singh", "rohan@example.com", "9876534567"),
        ("C6", "Ananya Gupta", "ananya@example.com", "9876545678")
    ]

    accounts_data = [
        ("A1", "C1", "ACC1001", 48300.00),
        ("A2", "C2", "ACC1002", 28300.00),
        ("A3", "C3", "ACC1003", 43500.00),
        ("A4", "C4", "ACC1004", 0.00),
        ("A5", "C5", "ACC1005", 55300.00),
        ("A6", "C6", "ACC1006", 36500.00)
    ]

    transactions_data = [
        # Aarav
        ("T1", "A1", "credit", 50000.00, "Salary credited", "2026-05-01 10:00:00"),
        ("T2", "A1", "debit", 2500.00, "Online shopping", "2026-05-02 14:30:00"),
        ("T3", "A1", "debit", 1200.00, "Electricity bill", "2026-05-03 09:15:00"),
        ("T4", "A1", "credit", 5000.00, "Refund received", "2026-05-04 16:45:00"),
        ("T5", "A1", "debit", 3000.00, "Restaurant", "2026-05-05 20:10:00"),

        # Rahul
        ("T6", "A2", "credit", 30000.00, "Salary credited", "2026-05-01 10:10:00"),
        ("T7", "A2", "debit", 1500.00, "Mobile recharge", "2026-05-03 12:00:00"),
        ("T8", "A2", "debit", 2200.00, "Groceries", "2026-05-04 18:20:00"),
        ("T9", "A2", "credit", 2000.00, "Cashback", "2026-05-05 11:00:00"),

        # Priya
        ("T10", "A3", "credit", 45000.00, "Salary credited", "2026-05-01 10:20:00"),
        ("T11", "A3", "debit", 2200.00, "Grocery purchase", "2026-05-04 18:30:00"),
        ("T12", "A3", "debit", 1500.00, "Uber rides", "2026-05-05 09:00:00"),
        ("T13", "A3", "credit", 3000.00, "Freelance payment", "2026-05-06 13:15:00"),
        ("T14", "A3", "debit", 800.00, "Coffee shop", "2026-05-06 18:45:00"),

        # Rohan
        ("T15", "A5", "credit", 60000.00, "Salary credited", "2026-05-01 09:30:00"),
        ("T16", "A5", "debit", 5000.00, "Flight booking", "2026-05-02 08:45:00"),
        ("T17", "A5", "debit", 2500.00, "Hotel booking", "2026-05-03 21:10:00"),
        ("T18", "A5", "credit", 4000.00, "Bonus", "2026-05-05 10:00:00"),
        ("T19", "A5", "debit", 1200.00, "Fuel", "2026-05-06 08:30:00"),

        # Ananya
        ("T20", "A6", "credit", 40000.00, "Salary credited", "2026-05-01 10:50:00"),
        ("T21", "A6", "debit", 3000.00, "Shopping", "2026-05-05 17:10:00"),
        ("T22", "A6", "debit", 2000.00, "Gym membership", "2026-05-06 07:30:00"),
        ("T23", "A6", "credit", 1500.00, "Cashback", "2026-05-06 14:00:00")
    ]

    # Insert data
    cursor.executemany("""
    INSERT OR IGNORE INTO customers
    (customer_id, name, email, phone)
    VALUES (?, ?, ?, ?);
    """, customers_data)

    cursor.executemany("""
    INSERT OR IGNORE INTO accounts
    (account_id, customer_id, account_number, balance)
    VALUES (?, ?, ?, ?);
    """, accounts_data)

    cursor.executemany("""
    INSERT OR IGNORE INTO transactions
    (transaction_id, account_id, transaction_type, amount, description, transaction_timestamp)
    VALUES (?, ?, ?, ?, ?, ?);
    """, transactions_data)

    # Commit changes
    conn.commit()

    print("Database created and populated successfully!")

    # Verify tables
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    print("Tables:", cursor.fetchall())

    conn.close()


init_db()

Database created and populated successfully!
Tables: [('customers',), ('accounts',), ('transactions',)]


In [6]:
# Function that executes a SQL query on the SQLite database and returns the result set

def sql_query(query):
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute(query)
    return cursor.fetchall()

In [7]:
sql_query("SELECT * FROM customers")

[('C1', 'Aarav Sharma', 'aarav@example.com', '9876543210'),
 ('C2', 'Rahul Verma', 'rahul@example.com', '9876501234'),
 ('C3', 'Priya Mehta', 'priya@example.com', '9876512345'),
 ('C4', 'Neha Kapoor', 'neha@example.com', '9876523456'),
 ('C5', 'Rohan Singh', 'rohan@example.com', '9876534567'),
 ('C6', 'Ananya Gupta', 'ananya@example.com', '9876545678')]

In [8]:
sql_query("SELECT * FROM accounts")

[('A1', 'C1', 'ACC1001', 48300.0),
 ('A2', 'C2', 'ACC1002', 28300.0),
 ('A3', 'C3', 'ACC1003', 43500.0),
 ('A4', 'C4', 'ACC1004', 0.0),
 ('A5', 'C5', 'ACC1005', 55300.0),
 ('A6', 'C6', 'ACC1006', 36500.0)]

In [9]:
sql_query("SELECT * FROM transactions")

[('T1', 'A1', 'credit', 50000.0, 'Salary credited', '2026-05-01 10:00:00'),
 ('T2', 'A1', 'debit', 2500.0, 'Online shopping', '2026-05-02 14:30:00'),
 ('T3', 'A1', 'debit', 1200.0, 'Electricity bill', '2026-05-03 09:15:00'),
 ('T4', 'A1', 'credit', 5000.0, 'Refund received', '2026-05-04 16:45:00'),
 ('T5', 'A1', 'debit', 3000.0, 'Restaurant', '2026-05-05 20:10:00'),
 ('T6', 'A2', 'credit', 30000.0, 'Salary credited', '2026-05-01 10:10:00'),
 ('T7', 'A2', 'debit', 1500.0, 'Mobile recharge', '2026-05-03 12:00:00'),
 ('T8', 'A2', 'debit', 2200.0, 'Groceries', '2026-05-04 18:20:00'),
 ('T9', 'A2', 'credit', 2000.0, 'Cashback', '2026-05-05 11:00:00'),
 ('T10', 'A3', 'credit', 45000.0, 'Salary credited', '2026-05-01 10:20:00'),
 ('T11', 'A3', 'debit', 2200.0, 'Grocery purchase', '2026-05-04 18:30:00'),
 ('T12', 'A3', 'debit', 1500.0, 'Uber rides', '2026-05-05 09:00:00'),
 ('T13', 'A3', 'credit', 3000.0, 'Freelance payment', '2026-05-06 13:15:00'),
 ('T14', 'A3', 'debit', 800.0, 'Coffee shop'

In [10]:
sql_query("""SELECT *
FROM transactions
WHERE DATE(transaction_timestamp) = '2026-05-05'
ORDER BY transaction_timestamp DESC;""")

[('T5', 'A1', 'debit', 3000.0, 'Restaurant', '2026-05-05 20:10:00'),
 ('T21', 'A6', 'debit', 3000.0, 'Shopping', '2026-05-05 17:10:00'),
 ('T9', 'A2', 'credit', 2000.0, 'Cashback', '2026-05-05 11:00:00'),
 ('T18', 'A5', 'credit', 4000.0, 'Bonus', '2026-05-05 10:00:00'),
 ('T12', 'A3', 'debit', 1500.0, 'Uber rides', '2026-05-05 09:00:00')]

In [11]:
sql_query("""SELECT *
FROM transactions
WHERE TIME(transaction_timestamp) BETWEEN '14:00:00' AND '17:00:00'
ORDER BY transaction_timestamp DESC;""")

[('T23', 'A6', 'credit', 1500.0, 'Cashback', '2026-05-06 14:00:00'),
 ('T4', 'A1', 'credit', 5000.0, 'Refund received', '2026-05-04 16:45:00'),
 ('T2', 'A1', 'debit', 2500.0, 'Online shopping', '2026-05-02 14:30:00')]

# **Part-A** Designing Tools

### **1. Tool to Check Account Balance** 

This tool is responsible for retrieving the current account balance of a customer using their `customer_id`. It connects to the database, fetches the corresponding account details, and returns a formatted response. The tool must handle both valid and invalid inputs while ensuring proper database interaction and clean output formatting.

**Steps to implement:**

* Accept `customer_id` as input from the user/agent
* Establish a connection to the SQLite database
* Execute a parameterized SQL query to fetch `account_number` and `balance`
* Check if a matching record exists

  * If found → return account number and balance
  * If not found → return an appropriate error message
* Print logs for tool invocation, input parameters, and output (for debugging)
* Close the database connection after execution


1. Check Account Balance

Create function to check the account balance
def check_account_balance(customer_id: str):

    # YOUR CODE HERE...

In [12]:
def check_account_balance(customer_id: str):
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    try:
        # Get account_id from customer_id
        cursor.execute("SELECT account_id FROM accounts WHERE customer_id = ?", (customer_id,))
        account_id_result = cursor.fetchone()

        if not account_id_result:
            print(f"ERROR: Customer ID {customer_id} not found.")
            return None

        account_id = account_id_result[0]

        # Fetch account number and balance using the account_id
        cursor.execute("SELECT account_number, balance FROM accounts WHERE account_id = ?", (account_id,))
        result = cursor.fetchone()

        if result:
            account_number, balance = result
            print(f"Tool Invoked: check_account_balance with customer_id={customer_id}")
            print(f"Output: Account Number: {account_number}, Balance: {balance:.2f}")
            return {"account_number": account_number, "balance": balance}
        else:
            print(f"ERROR: No account found for customer ID {customer_id}.")
            return None
    except sqlite3.Error as e:
        print(f"Database error: {e}")
        return None
    finally:
        conn.close()

In [13]:
# Test the function for a valid customer id
check_account_balance("C6")

Tool Invoked: check_account_balance with customer_id=C6
Output: Account Number: ACC1006, Balance: 36500.00


{'account_number': 'ACC1006', 'balance': 36500.0}

In [14]:
# Test the function for an invalid customer id
check_account_balance("C12")

ERROR: Customer ID C12 not found.


In [19]:
# Create a Tool for checking account balance, to be used by the agent for tool calling

# Use Structured Tool for multi-parameter function
from langchain_core.tools import StructuredTool

check_account_balance_tool = StructuredTool.from_function(
    name="check_account_balance",
    description="Checks the account balance for a given customer ID. Input is customer_id (str).",
    func=check_account_balance
)

In [20]:
# Test the tool for a valid customer id
check_account_balance_tool.run({
    "customer_id": "C6"
})

Tool Invoked: check_account_balance with customer_id=C6
Output: Account Number: ACC1006, Balance: 36500.00


{'account_number': 'ACC1006', 'balance': 36500.0}

In [21]:
# Test the tool for an invalid customer id
check_account_balance_tool.run({
    "customer_id": "C12"
})

ERROR: Customer ID C12 not found.


### **2. Tool to View Transactions** 

This tool retrieves the recent transaction history for a given customer based on the provided `customer_id`. It ensures that the customer exists, fetches the latest transactions from the database, and returns them in a structured tabular format. The tool also handles cases where the customer is invalid or has no transaction history.

**Steps to implement:**

* Accept `customer_id` and optional `limit` (default = 10) as input
* Establish a connection to the SQLite database
* Validate whether the given `customer_id` exists in the `customers` table

  * If not → return an error message
* Execute a SQL query to fetch recent transactions by joining `accounts` and `transactions` tables
* Sort transactions in descending order based on timestamp and limit the results
* Check if any transactions are returned

  * If none → return a “No transactions found” message
* Format the transaction data into a readable tabular structure
* Print logs for tool invocation, parameters, and output (for debugging)
* Close the database connection after execution


In [22]:
# 2. View Transactions

# Create function to view transactions
def view_transactions(customer_id: str, limit: int = 10):
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    try:
        # Validate customer_id exists
        cursor.execute("SELECT customer_id FROM customers WHERE customer_id = ?", (customer_id,))
        if not cursor.fetchone():
            print(f"ERROR: Customer ID {customer_id} not found.")
            return None

        # Get account_id for the customer
        cursor.execute("SELECT account_id FROM accounts WHERE customer_id = ?", (customer_id,))
        account_id_result = cursor.fetchone()

        if not account_id_result:
            print(f"ERROR: No account found for customer ID {customer_id}.")
            return None

        account_id = account_id_result[0]

        # Fetch recent transactions
        query = """
        SELECT transaction_id, transaction_type, amount, description, transaction_timestamp
        FROM transactions
        WHERE account_id = ?
        ORDER BY transaction_timestamp DESC
        LIMIT ?;
        """
        cursor.execute(query, (account_id, limit))
        transactions = cursor.fetchall()

        print(f"Tool Invoked: view_transactions with customer_id={customer_id}, limit={limit}")

        if not transactions:
            print(f"Output: No transactions found for customer ID {customer_id}.")
            return f"No transactions found for customer ID {customer_id}."

        # Format output into a readable tabular structure
        header = ["ID", "Type", "Amount", "Description", "Timestamp"]
        formatted_transactions = [header]
        for t_id, t_type, amount, desc, timestamp in transactions:
            formatted_transactions.append([t_id, t_type, f"{amount:.2f}", desc, timestamp])

        # Convert to a string representation for display
        max_lens = [max(len(str(item)) for item in col) for col in zip(*formatted_transactions)]
        output_str = "\n"
        for i, row in enumerate(formatted_transactions):
            output_str += " | ".join(str(item).ljust(max_lens[j]) for j, item in enumerate(row)) + "\n"
            if i == 0: # Add separator after header
                output_str += "-" * (sum(max_lens) + (len(max_lens) - 1) * 3) + "\n"

        print("Output:\n" + output_str)
        return output_str

    except sqlite3.Error as e:
        print(f"Database error: {e}")
        return None
    finally:
        conn.close()

In [ ]:
# Test the function for different scenarios

# YOUR CODE HERE...

In [23]:
# Test the function for a valid customer id with default limit
print("\n--- Testing with valid customer C1 (default limit) ---")
view_transactions("C1")

# Test the function for a valid customer id with a custom limit
print("\n--- Testing with valid customer C3 (limit 2) ---")
view_transactions("C3", limit=2)

# Test the function for an invalid customer id
print("\n--- Testing with invalid customer C99 ---")
view_transactions("C99")

# Test the function for a customer with no transactions (e.g., C4 has 0 balance)
print("\n--- Testing with customer C4 (expected no transactions) ---")
view_transactions("C4")


--- Testing with valid customer C1 (default limit) ---
Tool Invoked: view_transactions with customer_id=C1, limit=10
Output:

ID | Type   | Amount   | Description      | Timestamp          
---------------------------------------------------------------
T5 | debit  | 3000.00  | Restaurant       | 2026-05-05 20:10:00
T4 | credit | 5000.00  | Refund received  | 2026-05-04 16:45:00
T3 | debit  | 1200.00  | Electricity bill | 2026-05-03 09:15:00
T2 | debit  | 2500.00  | Online shopping  | 2026-05-02 14:30:00
T1 | credit | 50000.00 | Salary credited  | 2026-05-01 10:00:00


--- Testing with valid customer C3 (limit 2) ---
Tool Invoked: view_transactions with customer_id=C3, limit=2
Output:

ID  | Type   | Amount  | Description       | Timestamp          
----------------------------------------------------------------
T14 | debit  | 800.00  | Coffee shop       | 2026-05-06 18:45:00
T13 | credit | 3000.00 | Freelance payment | 2026-05-06 13:15:00


--- Testing with invalid customer C99 ---


'No transactions found for customer ID C4.'

In [24]:
# Create a Tool for viewing recent transactions, to be used by the agent for tool calling

# Use Structured Tool for multi-parameter function
from langchain_core.tools import StructuredTool

view_transactions_tool = StructuredTool.from_function(
    name="view_transactions",
    description="Retrieves recent transaction history for a given customer. Input customer_id (str) and optional limit (int, default=10).",
    func=view_transactions
)

In [25]:
# Test the tool for a valid customer id with default limit
print("\n--- Testing view_transactions_tool with C1 (default limit) ---")
view_transactions_tool.run({
    "customer_id": "C1"
})

# Test the tool for a valid customer id with a custom limit
print("\n--- Testing view_transactions_tool with C3 (limit 2) ---")
view_transactions_tool.run({
    "customer_id": "C3",
    "limit": 2
})

# Test the tool for an invalid customer id
print("\n--- Testing view_transactions_tool with invalid customer C99 ---")
view_transactions_tool.run({
    "customer_id": "C99"
})

# Test the tool for a customer with no transactions (e.g., C4 has 0 balance)
print("\n--- Testing view_transactions_tool with customer C4 (expected no transactions) ---")
view_transactions_tool.run({
    "customer_id": "C4"
})


--- Testing view_transactions_tool with C1 (default limit) ---
Tool Invoked: view_transactions with customer_id=C1, limit=10
Output:

ID | Type   | Amount   | Description      | Timestamp          
---------------------------------------------------------------
T5 | debit  | 3000.00  | Restaurant       | 2026-05-05 20:10:00
T4 | credit | 5000.00  | Refund received  | 2026-05-04 16:45:00
T3 | debit  | 1200.00  | Electricity bill | 2026-05-03 09:15:00
T2 | debit  | 2500.00  | Online shopping  | 2026-05-02 14:30:00
T1 | credit | 50000.00 | Salary credited  | 2026-05-01 10:00:00


--- Testing view_transactions_tool with C3 (limit 2) ---
Tool Invoked: view_transactions with customer_id=C3, limit=2
Output:

ID  | Type   | Amount  | Description       | Timestamp          
----------------------------------------------------------------
T14 | debit  | 800.00  | Coffee shop       | 2026-05-06 18:45:00
T13 | credit | 3000.00 | Freelance payment | 2026-05-06 13:15:00


--- Testing view_transacti

'No transactions found for customer ID C4.'

### **3. Tool to Filter Transactions Debit/Credit** 

This tool retrieves filtered transaction history for a given customer based on the specified `transaction_type` (debit or credit). It validates the input parameters, ensures the customer exists, and returns the filtered transactions in a structured tabular format. The tool also handles invalid inputs and cases where no transactions are found.

**Steps to implement:**

* Accept `customer_id`, `transaction_type` (debit/credit), and optional `limit` (default = 10) as input
* Convert `transaction_type` to lowercase and validate it

  * If invalid → return an error message
* Establish a connection to the SQLite database
* Validate whether the given `customer_id` exists in the `customers` table

  * If not → return an error message
* Execute a SQL query to fetch transactions by joining `accounts` and `transactions` tables

  * Apply filter on `transaction_type`
  * Sort by timestamp in descending order and limit the results
* Check if any transactions are returned

  * If none → return a “No transactions found” message
* Format the filtered transactions into a readable tabular structure
* Print logs for tool invocation, parameters, and output (for debugging)
* Close the database connection after execution


In [26]:
# 3. Filter Transactions Debit/Credit

# Create function to filter transactions
def filter_transactions(customer_id: str, transaction_type: str, limit: int = 10):
    transaction_type = transaction_type.lower()
    if transaction_type not in ['debit', 'credit']:
        print(f"ERROR: Invalid transaction type '{transaction_type}'. Must be 'debit' or 'credit'.")
        return None

    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    try:
        # Validate customer_id exists
        cursor.execute("SELECT customer_id FROM customers WHERE customer_id = ?", (customer_id,))
        if not cursor.fetchone():
            print(f"ERROR: Customer ID {customer_id} not found.")
            return None

        # Get account_id for the customer
        cursor.execute("SELECT account_id FROM accounts WHERE customer_id = ?", (customer_id,))
        account_id_result = cursor.fetchone()

        if not account_id_result:
            print(f"ERROR: No account found for customer ID {customer_id}.")
            return None

        account_id = account_id_result[0]

        # Fetch filtered transactions
        query = """
        SELECT transaction_id, transaction_type, amount, description, transaction_timestamp
        FROM transactions
        WHERE account_id = ? AND transaction_type = ?
        ORDER BY transaction_timestamp DESC
        LIMIT ?;
        """
        cursor.execute(query, (account_id, transaction_type, limit))
        transactions = cursor.fetchall()

        print(f"Tool Invoked: filter_transactions with customer_id={customer_id}, transaction_type={transaction_type}, limit={limit}")

        if not transactions:
            print(f"Output: No {transaction_type} transactions found for customer ID {customer_id}.")
            return f"No {transaction_type} transactions found for customer ID {customer_id}."

        # Format output into a readable tabular structure
        header = ["ID", "Type", "Amount", "Description", "Timestamp"]
        formatted_transactions = [header]
        for t_id, t_type, amount, desc, timestamp in transactions:
            formatted_transactions.append([t_id, t_type, f"{amount:.2f}", desc, timestamp])

        max_lens = [max(len(str(item)) for item in col) for col in zip(*formatted_transactions)]
        output_str = "\n"
        for i, row in enumerate(formatted_transactions):
            output_str += " | ".join(str(item).ljust(max_lens[j]) for j, item in enumerate(row)) + "\n"
            if i == 0:
                output_str += "-" * (sum(max_lens) + (len(max_lens) - 1) * 3) + "\n"

        print("Output:\n" + output_str)
        return output_str

    except sqlite3.Error as e:
        print(f"Database error: {e}")
        return None
    finally:
        conn.close()

In [27]:
# Test the function for different scenarios

# Test with customer C1 for debit transactions (default limit)
print("\n--- Testing C1 for debit transactions (default limit) ---")
filter_transactions("C1", "debit")

# Test with customer C3 for credit transactions with a custom limit
print("\n--- Testing C3 for credit transactions (limit 1) ---")
filter_transactions("C3", "credit", limit=1)

# Test with customer C5 for debit transactions (default limit)
print("\n--- Testing C5 for debit transactions (default limit) ---")
filter_transactions("C5", "debit")

# Test with customer C4 (no transactions, so no debit/credit)
print("\n--- Testing C4 for debit transactions (expected no transactions) ---")
filter_transactions("C4", "debit")

# Test with an invalid transaction type
print("\n--- Testing with invalid transaction type 'transfer' ---")
filter_transactions("C1", "transfer")


--- Testing C1 for debit transactions (default limit) ---
Tool Invoked: filter_transactions with customer_id=C1, transaction_type=debit, limit=10
Output:

ID | Type  | Amount  | Description      | Timestamp          
-------------------------------------------------------------
T5 | debit | 3000.00 | Restaurant       | 2026-05-05 20:10:00
T3 | debit | 1200.00 | Electricity bill | 2026-05-03 09:15:00
T2 | debit | 2500.00 | Online shopping  | 2026-05-02 14:30:00


--- Testing C3 for credit transactions (limit 1) ---
Tool Invoked: filter_transactions with customer_id=C3, transaction_type=credit, limit=1
Output:

ID  | Type   | Amount  | Description       | Timestamp          
----------------------------------------------------------------
T13 | credit | 3000.00 | Freelance payment | 2026-05-06 13:15:00


--- Testing C5 for debit transactions (default limit) ---
Tool Invoked: filter_transactions with customer_id=C5, transaction_type=debit, limit=10
Output:

ID  | Type  | Amount  | Descri

In [29]:
from langchain_core.tools import StructuredTool

filter_transactions_tool = StructuredTool.from_function(
    name="filter_transactions",
    description="Retrieves filtered transaction history for a given customer based on transaction_type (debit/credit). Input customer_id (str), transaction_type (str, 'debit' or 'credit'), and optional limit (int, default=10).",
    func=filter_transactions
)

In [ ]:
# Test the tool for different scenarios

# YOUR CODE HERE...

In [30]:
print("\n--- Testing filter_transactions_tool with C1 for debit transactions (default limit) ---")
filter_transactions_tool.run({
    "customer_id": "C1",
    "transaction_type": "debit"
})

print("\n--- Testing filter_transactions_tool with C3 for credit transactions (limit 1) ---")
filter_transactions_tool.run({
    "customer_id": "C3",
    "transaction_type": "credit",
    "limit": 1
})

print("\n--- Testing filter_transactions_tool with C5 for debit transactions (default limit) ---")
filter_transactions_tool.run({
    "customer_id": "C5",
    "transaction_type": "debit"
})

print("\n--- Testing filter_transactions_tool with C4 for debit transactions (expected no transactions) ---")
filter_transactions_tool.run({
    "customer_id": "C4",
    "transaction_type": "debit"
})

print("\n--- Testing filter_transactions_tool with invalid transaction type 'transfer' ---")
filter_transactions_tool.run({
    "customer_id": "C1",
    "transaction_type": "transfer"
})


--- Testing filter_transactions_tool with C1 for debit transactions (default limit) ---
Tool Invoked: filter_transactions with customer_id=C1, transaction_type=debit, limit=10
Output:

ID | Type  | Amount  | Description      | Timestamp          
-------------------------------------------------------------
T5 | debit | 3000.00 | Restaurant       | 2026-05-05 20:10:00
T3 | debit | 1200.00 | Electricity bill | 2026-05-03 09:15:00
T2 | debit | 2500.00 | Online shopping  | 2026-05-02 14:30:00


--- Testing filter_transactions_tool with C3 for credit transactions (limit 1) ---
Tool Invoked: filter_transactions with customer_id=C3, transaction_type=credit, limit=1
Output:

ID  | Type   | Amount  | Description       | Timestamp          
----------------------------------------------------------------
T13 | credit | 3000.00 | Freelance payment | 2026-05-06 13:15:00


--- Testing filter_transactions_tool with C5 for debit transactions (default limit) ---
Tool Invoked: filter_transactions wit

### **4. Tool to Transfer Money** 

This tool enables transferring money from one customer’s account to another account using the provided sender customer ID, receiver account number, and transfer amount. It performs necessary validations such as checking sender account existence, receiver account validity, sufficient balance, and positive transfer amount. The tool updates account balances, records transactions, and ensures data consistency using database operations.

**Steps to implement:**

* Accept `sender_customer_id`, `receiver_account_number`, and `amount` as input
* Validate that the transfer `amount` is greater than zero

  * If invalid → return an error message
* Establish a connection to the SQLite database
* Fetch sender account details using `customer_id`

  * If not found → return an error message
* Fetch receiver account details using `account_number`

  * If not found → return an error message
* Check if sender has sufficient balance

  * If insufficient → return an error message
* Generate current timestamp for transaction records
* Debit the sender’s account and credit the receiver’s account
* Insert corresponding debit and credit entries into the `transactions` table
* Commit the transaction to ensure changes are saved
* Handle exceptions using rollback in case of errors
* Print logs for tool invocation, parameters, and output (for debugging)
* Close the database connection after execution


In [31]:
def transfer_money(sender_customer_id: str, receiver_account_number: str, amount: float):
    # 1. Validate that the transfer amount is greater than zero
    if amount <= 0:
        print(f"ERROR: Transfer amount must be greater than zero. Received: {amount}")
        return None

    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    try:
        # Start a transaction
        conn.isolation_level = None # Autocommit off
        cursor.execute("BEGIN;")

        # 2. Fetch sender account details using customer_id
        cursor.execute("SELECT account_id, account_number, balance FROM accounts WHERE customer_id = ?", (sender_customer_id,))
        sender_account_details = cursor.fetchone()

        if not sender_account_details:
            print(f"ERROR: Sender Customer ID {sender_customer_id} not found or has no linked account.")
            cursor.execute("ROLLBACK;")
            return None

        sender_account_id, sender_account_number, sender_balance = sender_account_details

        # 3. Fetch receiver account details using account_number
        cursor.execute("SELECT account_id, customer_id, balance FROM accounts WHERE account_number = ?", (receiver_account_number,))
        receiver_account_details = cursor.fetchone()

        if not receiver_account_details:
            print(f"ERROR: Receiver Account Number {receiver_account_number} not found.")
            cursor.execute("ROLLBACK;")
            return None

        receiver_account_id, receiver_customer_id, receiver_balance = receiver_account_details

        # Prevent transferring to self
        if sender_account_id == receiver_account_id:
            print("ERROR: Cannot transfer money to the same account.")
            cursor.execute("ROLLBACK;")
            return None

        # 4. Check if sender has sufficient balance
        if sender_balance < amount:
            print(f"ERROR: Insufficient balance. Sender {sender_customer_id} has {sender_balance:.2f}, but tried to transfer {amount:.2f}.")
            cursor.execute("ROLLBACK;")
            return None

        # 5. Generate current timestamp for transaction records
        transaction_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        print(f"Tool Invoked: transfer_money with sender_customer_id={sender_customer_id}, receiver_account_number={receiver_account_number}, amount={amount}")

        # 6. Debit the sender’s account
        new_sender_balance = sender_balance - amount
        cursor.execute("UPDATE accounts SET balance = ? WHERE account_id = ?", (new_sender_balance, sender_account_id))

        # Insert debit transaction for sender
        cursor.execute(
            "INSERT INTO transactions (transaction_id, account_id, transaction_type, amount, description, transaction_timestamp) VALUES (?, ?, ?, ?, ?, ?)",
            (f"TRF_DEB_{sender_account_id}_{datetime.now().microsecond}", sender_account_id, "debit", amount, f"Transfer to {receiver_account_number}", transaction_timestamp)
        )

        # 7. Credit the receiver’s account
        new_receiver_balance = receiver_balance + amount
        cursor.execute("UPDATE accounts SET balance = ? WHERE account_id = ?", (new_receiver_balance, receiver_account_id))

        # Insert credit transaction for receiver
        cursor.execute(
            "INSERT INTO transactions (transaction_id, account_id, transaction_type, amount, description, transaction_timestamp) VALUES (?, ?, ?, ?, ?, ?)",
            (f"TRF_CRD_{receiver_account_id}_{datetime.now().microsecond}", receiver_account_id, "credit", amount, f"Transfer from {sender_account_number}", transaction_timestamp)
        )

        # Commit the transaction
        cursor.execute("COMMIT;")
        print(f"Output: Successfully transferred {amount:.2f} from {sender_customer_id}'s account ({sender_account_number}) to {receiver_account_number}.")
        print(f"New balances: Sender {sender_account_number}: {new_sender_balance:.2f}, Receiver {receiver_account_number}: {new_receiver_balance:.2f}")
        return f"Successfully transferred {amount:.2f} from {sender_customer_id}'s account ({sender_account_number}) to {receiver_account_number}."

    except sqlite3.Error as e:
        print(f"Database error during transfer: {e}")
        cursor.execute("ROLLBACK;")
        return None
    finally:
        conn.close()

In [32]:
# Test the function for different scenarios

print("--- Initial Balances ---")
check_account_balance("C1") # ACC1001
check_account_balance("C2") # ACC1002
check_account_balance("C4") # ACC1004 (balance 0)

# Scenario 1: Successful transfer
print("\n--- Scenario 1: Successful transfer (C1 to ACC1002, 1000) ---")
transfer_money("C1", "ACC1002", 1000.00)
check_account_balance("C1")
check_account_balance("C2")

# Scenario 2: Insufficient balance
print("\n--- Scenario 2: Insufficient balance (C1 to ACC1004, 50000) ---")
transfer_money("C1", "ACC1004", 50000.00)
check_account_balance("C1")
check_account_balance("C4")

# Scenario 3: Invalid sender customer ID
print("\n--- Scenario 3: Invalid sender customer ID ('C99' to ACC1002, 100) ---")
transfer_money("C99", "ACC1002", 100.00)

# Scenario 4: Invalid receiver account number
print("\n--- Scenario 4: Invalid receiver account number (C1 to 'ACC9999', 100) ---")
transfer_money("C1", "ACC9999", 100.00)

# Scenario 5: Transfer amount is zero or negative
print("\n--- Scenario 5: Transfer amount is zero (C1 to ACC1002, 0) ---")
transfer_money("C1", "ACC1002", 0.00)

print("\n--- Scenario 6: Transfer amount is negative (C1 to ACC1002, -50) ---")
transfer_money("C1", "ACC1002", -50.00)

# Scenario 7: Transfer to self (ACC1001 is C1's account)
print("\n--- Scenario 7: Transfer to self (C1 to ACC1001, 100) ---")
transfer_money("C1", "ACC1001", 100.00)


print("\n--- Final Balances After All Tests ---")
check_account_balance("C1")
check_account_balance("C2")
check_account_balance("C4")

--- Initial Balances ---
Tool Invoked: check_account_balance with customer_id=C1
Output: Account Number: ACC1001, Balance: 48300.00
Tool Invoked: check_account_balance with customer_id=C2
Output: Account Number: ACC1002, Balance: 28300.00
Tool Invoked: check_account_balance with customer_id=C4
Output: Account Number: ACC1004, Balance: 0.00

--- Scenario 1: Successful transfer (C1 to ACC1002, 1000) ---
Tool Invoked: transfer_money with sender_customer_id=C1, receiver_account_number=ACC1002, amount=1000.0
Output: Successfully transferred 1000.00 from C1's account (ACC1001) to ACC1002.
New balances: Sender ACC1001: 47300.00, Receiver ACC1002: 29300.00
Tool Invoked: check_account_balance with customer_id=C1
Output: Account Number: ACC1001, Balance: 47300.00
Tool Invoked: check_account_balance with customer_id=C2
Output: Account Number: ACC1002, Balance: 29300.00

--- Scenario 2: Insufficient balance (C1 to ACC1004, 50000) ---
ERROR: Insufficient balance. Sender C1 has 47300.00, but tried t

{'account_number': 'ACC1004', 'balance': 0.0}

In [33]:
from langchain_core.tools import StructuredTool

transfer_money_tool = StructuredTool.from_function(
    name="transfer_money",
    description="Transfers money from one customer's account to another. Input sender_customer_id (str), receiver_account_number (str), and amount (float).",
    func=transfer_money
)

In [35]:
# Test the tool for different scenarios

print("--- Initial Balances (Tool Test) ---")
check_account_balance_tool.run({"customer_id": "C1"})
check_account_balance_tool.run({"customer_id": "C2"})
check_account_balance_tool.run({"customer_id": "C4"})

# Scenario 1: Successful transfer
print("\n--- Scenario 1: Successful transfer (C1 to ACC1002, 500) ---")
transfer_money_tool.run({"sender_customer_id": "C1", "receiver_account_number": "ACC1002", "amount": 500.00})
check_account_balance_tool.run({"customer_id": "C1"})
check_account_balance_tool.run({"customer_id": "C2"})

# Scenario 2: Insufficient balance
print("\n--- Scenario 2: Insufficient balance (C1 to ACC1004, 100000) ---")
transfer_money_tool.run({"sender_customer_id": "C1", "receiver_account_number": "ACC1004", "amount": 100000.00})
check_account_balance_tool.run({"customer_id": "C1"})
check_account_balance_tool.run({"customer_id": "C4"})

# Scenario 3: Invalid sender customer ID
print("\n--- Scenario 3: Invalid sender customer ID ('C99' to ACC1002, 100) ---")
transfer_money_tool.run({"sender_customer_id": "C99", "receiver_account_number": "ACC1002", "amount": 100.00})

# Scenario 4: Invalid receiver account number
print("\n--- Scenario 4: Invalid receiver account number (C1 to 'ACC9999', 100) ---")
transfer_money_tool.run({"sender_customer_id": "C1", "receiver_account_number": "ACC9999", "amount": 100.00})

# Scenario 5: Transfer amount is zero or negative
print("\n--- Scenario 5: Transfer amount is zero (C1 to ACC1002, 0) ---")
transfer_money_tool.run({"sender_customer_id": "C1", "receiver_account_number": "ACC1002", "amount": 0.00})

print("\n--- Scenario 6: Transfer amount is negative (C1 to ACC1002, -50) ---")
transfer_money_tool.run({"sender_customer_id": "C1", "receiver_account_number": "ACC1002", "amount": -50.00})

# Scenario 7: Transfer to self (ACC1001 is C1's account) - this relies on having the ACC1001 in the system
# Note: This will only work if check_account_balance_tool was run and the account number for C1 was known.
# For a proper test, you might need to query the DB directly for ACC1001 or hardcode it.
print("\n--- Scenario 7: Transfer to self (C1 to ACC1001, 100) ---")
transfer_money_tool.run({"sender_customer_id": "C1", "receiver_account_number": "ACC1001", "amount": 100.00})


print("\n--- Final Balances After All Tool Tests ---")
check_account_balance_tool.run({"customer_id": "C1"})
check_account_balance_tool.run({"customer_id": "C2"})
check_account_balance_tool.run({"customer_id": "C4"})

--- Initial Balances (Tool Test) ---
Tool Invoked: check_account_balance with customer_id=C1
Output: Account Number: ACC1001, Balance: 46800.00
Tool Invoked: check_account_balance with customer_id=C2
Output: Account Number: ACC1002, Balance: 29800.00
Tool Invoked: check_account_balance with customer_id=C4
Output: Account Number: ACC1004, Balance: 0.00

--- Scenario 1: Successful transfer (C1 to ACC1002, 500) ---
Tool Invoked: transfer_money with sender_customer_id=C1, receiver_account_number=ACC1002, amount=500.0
Output: Successfully transferred 500.00 from C1's account (ACC1001) to ACC1002.
New balances: Sender ACC1001: 46300.00, Receiver ACC1002: 30300.00
Tool Invoked: check_account_balance with customer_id=C1
Output: Account Number: ACC1001, Balance: 46300.00
Tool Invoked: check_account_balance with customer_id=C2
Output: Account Number: ACC1002, Balance: 30300.00

--- Scenario 2: Insufficient balance (C1 to ACC1004, 100000) ---
ERROR: Insufficient balance. Sender C1 has 46300.00, b

{'account_number': 'ACC1004', 'balance': 0.0}

### **5. Tool to Generate Mini Statement**

This tool generates a mini statement for a given customer by retrieving recent transactions, calculating the running balance, and presenting the information in a structured, bank-style format. It validates the customer and account details, fetches the latest transactions, and formats the output to provide a clear snapshot of account activity.

**Steps to implement:**

* Accept `customer_id` and optional `limit` (default = 5) as input
* Establish a connection to the SQLite database
* Validate whether the given `customer_id` exists in the `customers` table

  * If not → return an error message
* Fetch account details (`account_id`, `account_number`, `balance`)

  * If not found → return an error message
* Mask the account number for secure display (eg. XXXX1006)
* Execute a SQL query to retrieve the latest transactions for the account

  * Sort by timestamp in descending order and limit the results
* Check if any transactions are returned

  * If none → return a “No transactions found” message
* Calculate running balance in reverse order based on current balance
* Format the output into a structured mini statement including:

  * Customer details
  * Account number (masked)
  * Generated timestamp
  * Transaction table with date, type, amount, balance, and description
  * Current balance
* Print logs for tool invocation, parameters, and output (for debugging)
* Close the database connection after execution


In [36]:
def generate_mini_statement(customer_id: str, limit: int = 5):
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    try:
        # Validate customer_id exists
        cursor.execute("SELECT customer_id, name FROM customers WHERE customer_id = ?", (customer_id,))
        customer_details = cursor.fetchone()
        if not customer_details:
            print(f"ERROR: Customer ID {customer_id} not found.")
            return None
        customer_name = customer_details[1]

        # Fetch account details (account_id, account_number, balance)
        cursor.execute("SELECT account_id, account_number, balance FROM accounts WHERE customer_id = ?", (customer_id,))
        account_details = cursor.fetchone()
        if not account_details:
            print(f"ERROR: No account found for customer ID {customer_id}.")
            return None
        account_id, account_number, current_balance = account_details

        # Mask the account number
        masked_account_number = "XXXX" + account_number[-4:]

        # Retrieve the latest transactions for the account
        query = """
        SELECT transaction_type, amount, description, transaction_timestamp
        FROM transactions
        WHERE account_id = ?
        ORDER BY transaction_timestamp DESC
        LIMIT ?;
        """
        cursor.execute(query, (account_id, limit))
        transactions = cursor.fetchall()

        print(f"Tool Invoked: generate_mini_statement with customer_id={customer_id}, limit={limit}")

        if not transactions:
            print(f"Output: No transactions found for customer ID {customer_id}. No mini statement generated.")
            return f"No transactions found for customer ID {customer_id}. No mini statement generated."

        # Calculate running balance in reverse order
        running_balances = []
        temp_balance = current_balance
        # Transactions are already in descending order by timestamp (most recent first)
        # To calculate running balance correctly, we need to process them in reverse chronological order
        # i.e., from oldest to newest within the selected limit.
        # So, we reverse the fetched transactions to simulate chronological processing
        transactions_chronological = list(reversed(transactions))

        for t_type, amount, _, _ in transactions_chronological:
            if t_type == 'debit':
                temp_balance += amount # To get the balance *before* this debit
            else:
                temp_balance -= amount # To get the balance *before* this credit
            running_balances.append(temp_balance)

        # Reverse running_balances to match the order of `transactions` (most recent first)
        running_balances.reverse()

        # Format the output into a structured mini statement
        statement_output = []
        statement_output.append("\n" + "*" * 50)
        statement_output.append(f"{'MINI STATEMENT':^50}")
        statement_output.append("*" * 50)
        statement_output.append(f"Customer Name: {customer_name}")
        statement_output.append(f"Account Number: {masked_account_number}")
        statement_output.append(f"Statement Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        statement_output.append("-" * 50)

        header = ["Date", "Type", "Amount", "Balance", "Description"]
        transaction_rows = [header]
        for i, (t_type, amount, desc, timestamp) in enumerate(transactions):
            # Extract only date for display
            display_date = timestamp.split(' ')[0]
            transaction_rows.append([display_date, t_type.capitalize(), f"{amount:.2f}", f"{running_balances[i]:.2f}", desc])

        # Determine max column widths for formatting
        max_lens = [max(len(str(item)) for item in col) for col in zip(*transaction_rows)]
        for i, row in enumerate(transaction_rows):
            statement_output.append(" | ".join(str(item).ljust(max_lens[j]) for j, item in enumerate(row)))
            if i == 0:
                statement_output.append("-" * (sum(max_lens) + (len(max_lens) - 1) * 3))

        statement_output.append("-" * 50)
        statement_output.append(f"Current Balance: {current_balance:.2f}")
        statement_output.append("*" * 50)

        output_str = "\n".join(statement_output)
        print("Output:\n" + output_str)
        return output_str

    except sqlite3.Error as e:
        print(f"Database error: {e}")
        return None
    finally:
        conn.close()

In [37]:
# Test the function

# Scenario 1: Valid customer with default limit (5 transactions)
print("\n--- Testing generate_mini_statement for C1 (default limit=5) ---")
generate_mini_statement("C1")

# Scenario 2: Valid customer with a custom limit (e.g., 3 transactions)
print("\n--- Testing generate_mini_statement for C3 (limit=3) ---")
generate_mini_statement("C3", limit=3)

# Scenario 3: Valid customer with no transactions (e.g., C4)
print("\n--- Testing generate_mini_statement for C4 (no transactions) ---")
generate_mini_statement("C4")

# Scenario 4: Invalid customer ID
print("\n--- Testing generate_mini_statement for C99 (invalid customer) ---")
generate_mini_statement("C99")

# Scenario 5: Valid customer with more transactions than the limit (e.g., C5, limit=2)
print("\n--- Testing generate_mini_statement for C5 (limit=2) ---")
generate_mini_statement("C5", limit=2)


--- Testing generate_mini_statement for C1 (default limit=5) ---
Tool Invoked: generate_mini_statement with customer_id=C1, limit=5
Output:

**************************************************
                  MINI STATEMENT                  
**************************************************
Customer Name: Aarav Sharma
Account Number: XXXX1001
Statement Date: 2026-05-10 03:16:57
--------------------------------------------------
Date       | Type   | Amount  | Balance  | Description        
--------------------------------------------------------------
2026-05-10 | Debit  | 500.00  | 46300.00 | Transfer to ACC1002
2026-05-10 | Debit  | 500.00  | 45800.00 | Transfer to ACC1002
2026-05-10 | Debit  | 1000.00 | 45300.00 | Transfer to ACC1002
2026-05-05 | Debit  | 3000.00 | 44300.00 | Restaurant         
2026-05-04 | Credit | 5000.00 | 41300.00 | Refund received    
--------------------------------------------------
Current Balance: 46300.00
***********************************************

'\n**************************************************\n                  MINI STATEMENT                  \n**************************************************\nCustomer Name: Rohan Singh\nAccount Number: XXXX1005\nStatement Date: 2026-05-10 03:16:57\n--------------------------------------------------\nDate       | Type   | Amount  | Balance  | Description\n------------------------------------------------------\n2026-05-06 | Debit  | 1200.00 | 52500.00 | Fuel       \n2026-05-05 | Credit | 4000.00 | 51300.00 | Bonus      \n--------------------------------------------------\nCurrent Balance: 55300.00\n**************************************************'

In [38]:
from langchain_core.tools import StructuredTool

generate_mini_statement_tool = StructuredTool.from_function(
    name="generate_mini_statement",
    description="Generates a mini statement for a given customer, retrieving recent transactions and calculating running balances. Input customer_id (str) and optional limit (int, default=5).",
    func=generate_mini_statement
)

In [ ]:
# Test the tool

# Scenario 1: Valid customer with default limit (5 transactions)
print("\n--- Testing generate_mini_statement_tool for C1 (default limit=5) ---")
generate_mini_statement_tool.run({"customer_id": "C1"})

# Scenario 2: Valid customer with a custom limit (e.g., 3 transactions)
print("\n--- Testing generate_mini_statement_tool for C3 (limit=3) ---")
generate_mini_statement_tool.run({"customer_id": "C3", "limit": 3})

# Scenario 3: Valid customer with no transactions (e.g., C4)
print("\n--- Testing generate_mini_statement_tool for C4 (no transactions) ---")
generate_mini_statement_tool.run({"customer_id": "C4"})

# Scenario 4: Invalid customer ID
print("\n--- Testing generate_mini_statement_tool for C99 (invalid customer) ---")
generate_mini_statement_tool.run({"customer_id": "C99"})

# Scenario 5: Valid customer with more transactions than the limit (e.g., C5, limit=2)
print("\n--- Testing generate_mini_statement_tool for C5 (limit=2) ---")
generate_mini_statement_tool.run({"customer_id": "C5", "limit": 2})

---
---

# **Part-B** Building Agent & Testing

## **Create Agent**

Define the complete Agentic Banking System by integrating all tools, configuring the system message, and building the agent workflow using a graph-based orchestration approach (LangGraph).

It enables the agent to interpret user queries, dynamically select tools, and manage multi-step interactions with memory support.

**Steps to implement:**

* Define a list of all banking tools (`check_account_balance`, `view_transactions`, `filter_transactions`, `transfer_money`, `generate_mini_statement`)
* Create a system message that clearly instructs the agent on:

  * Supported operations
  * Tool usage rules
  * Input handling
  * Error handling
  * Response formatting
* Initialize a ReAct-based agent using the LLM, tools list, and system prompt
* Define a `State` structure to manage conversation messages
    ```python
    from langgraph.graph.message import add_messages
    from typing import TypedDict, List, Optional, Annotated

    class State(TypedDict):
        messages: Annotated[list, add_messages]
    ```
* Build a graph workflow using `StateGraph`:

  * Add an **agent node** for decision-making
  * Add a **tool node** for executing tool calls
* Configure conditional edges:

  * Route to tools if required (`tools_condition`)
  * Return back to agent after tool execution
* Set the agent as the entry point of the workflow
* Initialize memory using `MemorySaver` to persist conversation context across interactions
    ```python
    from langgraph.checkpoint.memory import MemorySaver
    memory = MemorySaver()
    ```
* Compile the graph with memory support to enable stateful execution
    ```python
    .compile(checkpointer=memory)
    ```


In [ ]:
#@title Load the function `display_workflow_graph()`

## Render a LangGraph-style Mermaid string in Jupyter/Colab

from IPython.display import HTML, display
import json, re

def display_workflow_graph(workflow_app):
    mermaid_str = workflow_app.get_graph().draw_mermaid()
    s = str(mermaid_str).strip()
    cfg_json = "{}"

    # Extract optional YAML front-matter like:
    # ---
    # config:
    #   flowchart:
    #     curve: linear
    # ---
    m = re.match(r"^---\s*(.*?)\s*---\s*(graph\s+\w+;.*)$", s, re.S)
    if m:
        yaml_block, diagram = m.group(1), m.group(2)
        try:
            import yaml  # pip install pyyaml (only once if missing)
            data = yaml.safe_load(yaml_block) or {}
            cfg_json = json.dumps(data.get("config", {}))
        except Exception:
            diagram = s.split('---')[-1].strip()
    else:
        diagram = s

    # Convert YAML config to Mermaid init directive + embed and render
    directive = f"%%{{init: {cfg_json}}}%%\n"
    html = f"""
    <div class="mermaid">
    {directive}
    {diagram}
    </div>
    <script>
    (function() {{
      function boot() {{
        if (window.mermaid) {{
          mermaid.initialize({{ startOnLoad: true, securityLevel: 'loose' }});
          mermaid.contentLoaded();
        }} else {{
          var s = document.createElement('script');
          s.src = "https://cdn.jsdelivr.net/npm/mermaid/dist/mermaid.min.js";
          s.onload = function() {{
            mermaid.initialize({{ startOnLoad: true, securityLevel: 'loose' }});
            mermaid.contentLoaded();
          }};
          document.head.appendChild(s);
        }}
      }}
      boot();
    }})();
    </script>
    """
    display(HTML(html))

In [ ]:
# Display the workflow graph

# YOUR CODE HERE...

### **Test the Application**

This function serves as the interface to interact with the Agentic Banking System by sending user queries to the compiled graph and retrieving the agent’s response. It wraps the user input into a message format, invokes the graph workflow, and prints the final response generated by the agent after tool execution and reasoning.

**Steps to implement:**

* Accept user input (`human_message`) as a string
* Wrap the input using `HumanMessage` format required by LangChain
* Define a configuration object (`conf`) with a `thread_id` to maintain session continuity
    ```python
    conf = {"configurable": {"thread_id": "1"}}
    ```
* Invoke the compiled graph using the input message and configuration
    ```python
    response = banking_graph.invoke(
        {"messages": [HumanMessage(content=human_message)]}, conf
    )
    ```
* The graph processes the query through:

  * Agent reasoning
  * Tool selection (if required)
  * Tool execution
* Extract the final response from the last message in the response object
* Print a separator for readability in logs/output
* Display the agent’s final response to the user


In [ ]:
# YOUR CODE HERE...

In [ ]:
# YOUR CODE HERE...

In [ ]:
# YOUR CODE HERE...

In [ ]:
# YOUR CODE HERE...

## **Gradio Implementation**

**Gradio Chat Interface**

Build a user-friendly chat interface using Gradio to interact with the Agentic Banking System. It should connect the frontend chat UI with the backend agent graph, allowing users to submit queries and receive responses in real time.

**Steps to implement:**

* Define a function `get_response(message, history)` to handle user input
* Wrap the user message using `HumanMessage` and pass it to the `banking_graph.invoke()` method
* Use the predefined configuration (`conf`) to maintain session continuity
* Extract the agent’s final response from the returned messages
* Return the response to be displayed in the chat interface
* Create a Gradio `ChatInterface`:

  * Set the function (`fn`) to `get_response`
  * Provide a title and description for the application
  * Add example queries to guide users
  * Configure theme and message format (`type='messages'`)
        
    ```python
    gr.ChatInterface(
        fn=...,
        title=...,
        description=...,
        examples=[
            "What is my account balance? My customer id is C1",
            "Show my last 5 transactions. My customer id is C1",
            "Show only debit transactions for customer C1",
            "Transfer 1000 from customer C1 to account A2",
            "Generate my mini statement for customer C1"
        ],
        theme="soft",
        type='messages'
    )
    ```

* Launch the application using `demo.launch(debug=True)` for interactive testing


In [ ]:
import gradio as gr

In [ ]:
def get_response(message, history):

    # YOUR CODE HERE...


In [ ]:
# Create Gradio Chat Interface

# YOUR CODE HERE...

In [ ]:
# YOUR CODE HERE...

---

<center>
$END$
</center>

---